In [1]:
pip install mmh3 bitarray

Note: you may need to restart the kernel to use updated packages.


In [3]:
#standard bf
import mmh3
from bitarray import bitarray
import math
class BloomFilter:
    def __init__(self, n: int, fpr: float):
        #n is the expected number of elements
        #fpr is the desired false positive rate (like 0.01 for 1%)
        self.n = n
        self.fpr = fpr
        self.m = self._optimal_m(n, fpr)   #number of bits
        self.k = self._optimal_k(self.m, n) #number of hash functions
        self.bit_array = bitarray(self.m)
        self.bit_array.setall(0)

    def _optimal_m(self, n, fpr):
        return int(-n * math.log(fpr) / (math.log(2) ** 2))

    def _optimal_k(self, m, n):
        return max(1, int((m / n) * math.log(2)))

    def _hashes(self, key: str):
        #this will generate k different hash indices using mmh3 with different seeds
        return [mmh3.hash(key, seed=i, signed=False) % self.m for i in range(self.k)]

    def insert(self, key: str) -> None:
        for idx in self._hashes(key):
            self.bit_array[idx] = 1

    def query(self, key: str) -> bool:
        return all(self.bit_array[idx] for idx in self._hashes(key))

    def memory_bits(self) -> int:
        return self.m


In [4]:
#testing bf
import random
import string

def random_key(length=10):
    return ''.join(random.choices(string.ascii_lowercase, k=length))

def test_zero_false_negatives():
    print("Testing: zero false negatives...")
    bf = BloomFilter(n=10000, fpr=0.01)
    keys = [random_key() for _ in range(10000)]
    for k in keys:
        bf.insert(k)
    for k in keys:
        assert bf.query(k), f"False negative found on key: {k}"
    print("PASS: all 10,000 inserted keys found correctly\n")

def test_fpr():
    print("Testing: false positive rate...")
    target_fpr = 0.01
    bf = BloomFilter(n=10000, fpr=target_fpr)

    positives = [random_key() for _ in range(10000)]
    for k in positives:
        bf.insert(k)

    positive_set = set(positives)
    false_positives = 0
    trials = 100000

    for _ in range(trials):
        key = random_key(length=15)  # longer keys = almost certainly not inserted
        if key not in positive_set and bf.query(key):
            false_positives += 1

    actual_fpr = false_positives / trials
    print(f"Target FPR : {target_fpr:.3f}")
    print(f"Actual FPR : {actual_fpr:.4f}")
    assert actual_fpr < target_fpr * 2, "FPR is way off — check your formulas"
    print("PASS: FPR within acceptable range\n")

def test_memory():
    print("Testing: memory_bits()...")
    bf = BloomFilter(n=10000, fpr=0.01)
    bits = bf.memory_bits()
    assert bits > 0
    print(f"PASS: memory_bits() = {bits} bits ({bits // 8} bytes)\n")

test_zero_false_negatives()
test_fpr()
test_memory()

Testing: zero false negatives...
PASS: all 10,000 inserted keys found correctly

Testing: false positive rate...
Target FPR : 0.010
Actual FPR : 0.0100
PASS: FPR within acceptable range

Testing: memory_bits()...
PASS: memory_bits() = 95850 bits (11981 bytes)



In [5]:
#benchmarking bf
def random_key(length=10):
    return ''.join(random.choices(string.ascii_lowercase, k=length))

n = 10000
print(f"n = {n} elements\n")
print(f"{'Target FPR':>12} {'Bits (m)':>10} {'Bytes':>8} {'k (hashes)':>12} {'Actual FPR':>12}")
print("-" * 58)

for fpr in [0.001, 0.01, 0.05, 0.1]:
    bf = BloomFilter(n=n, fpr=fpr)

    keys = [random_key() for _ in range(n)]
    for k in keys:
        bf.insert(k)

    positive_set = set(keys)
    fp = sum(
        1 for _ in range(50000)
        if (k := random_key(length=15)) not in positive_set and bf.query(k)
    )
    actual = fp / 50000

    print(f"{fpr:>12.3f} {bf.memory_bits():>10} {bf.memory_bits()//8:>8} {bf.k:>12} {actual:>12.4f}")

n = 10000 elements

  Target FPR   Bits (m)    Bytes   k (hashes)   Actual FPR
----------------------------------------------------------
       0.001     143775    17971            9       0.0010
       0.010      95850    11981            6       0.0104
       0.050      62352     7794            4       0.0506
       0.100      47925     5990            3       0.0996
